> ## SOLUTION / ANSWER KEY &mdash; Lab 6.5
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-05-run-and-trace.ipynb`](../lab-05-run-and-trace.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.5 &mdash; Run the Agent & Read the Trace

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Read a real agent trace: the list of messages it returns
- Extract which tools were called (from each AIMessage.tool_calls)
- Cap the loop with recursion_limit so it can never run forever

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`). The **graded** cells assert only on the deterministic parts you build &mdash; tool wiring, prompt formatting, agent structure, routing and guardrails &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable.

**Reference:** [Module 6 slides &mdash; The agent loop, made visible](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-06-05"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
`create_agent` runs the loop for you: the model decides, tools run, results feed back &mdash; until a
final answer, or until **`recursion_limit`** stops it. The agent returns a **list of messages** (the
trace): **`AIMessage`** with **`.tool_calls`** (what it decided to run), **`ToolMessage`** (the result),
and a final `AIMessage` with **`.content`**. Reading that trace is deterministic &mdash; here you
practise on a fixed sample, then run a live agent in the optional cell.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# A real agent returns messages like these -- a fixed sample to practise reading:
SAMPLE = [
    HumanMessage(content="population of france divided by 2"),
    AIMessage(content="", tool_calls=[{"name": "web_search", "args": {"query": "population of france"}, "id": "a"}]),
    ToolMessage(content="68000000", tool_call_id="a"),
    AIMessage(content="", tool_calls=[{"name": "calculator", "args": {"expression": "68000000/2"}, "id": "b"}]),
    ToolMessage(content="34000000.0", tool_call_id="b"),
    AIMessage(content="The population halved is 34000000."),
]
print("messages in the trace:", len(SAMPLE))

## Your Turn
Complete `tools_used` (every tool call, in order), `final_answer` (the last message with text), and
`run_config` (the loop cap the agent honors).

In [ ]:
from langchain_core.messages import AIMessage

def tools_used(messages):
    names = []
    for m in messages:
        for tc in getattr(m, "tool_calls", None) or []:   # AIMessages carry tool_calls
            names.append(tc["name"])
    return names

def final_answer(messages):
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content:   # the last AIMessage that has text
            return m.content
    return None

def run_config(max_steps):
    return {"recursion_limit": max_steps}

try:
    print('tools used  :', tools_used(SAMPLE))
    print('final answer:', final_answer(SAMPLE))
    print('run config  :', run_config(8))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the trace shows both tools, in order", lambda: tools_used(SAMPLE) == ["web_search", "calculator"])
expect_true("a no-tool trace yields no tool calls", lambda: tools_used([AIMessage(content="hi")]) == [])
expect_true("the final answer is the model's last text", lambda: final_answer(SAMPLE) == "The population halved is 34000000.")
expect_true("run_config caps the loop with recursion_limit", lambda: run_config(3)["recursion_limit"] == 3)
expect_true("a smaller cap is a smaller number", lambda: run_config(1)["recursion_limit"] == 1)

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

## Optional &mdash; run it for real (not graded)
Run a real agent and read its trace with the very same functions. This calls a **real** local model via `ChatOllama("llama3.2:1b")` &mdash; start it with
`ollama run llama3.2:1b` (or swap in `ChatGroq` with a `GROQ_API_KEY`). If none is reachable the cell
prints a note and moves on. **The graded cells above never call an LLM, so the lab always verifies offline.**
*(llama3.2:1b is tiny &mdash; tool-calling can be hit-or-miss; the point is to see a real invocation.)*

In [ ]:
try:
    if ollama_up():
        from langchain_core.tools import tool
        from langchain_ollama import ChatOllama
        from langchain.agents import create_agent
        @tool
        def double(n: int) -> int:
            """Double an integer."""
            return n * 2
        agent = create_agent(ChatOllama(model="llama3.2:1b"), [double])
        result = agent.invoke({"messages": [{"role": "user", "content": "Use the double tool on 21."}]},
                              config=run_config(8))
        print("tools used live:", tools_used(result["messages"]))
        print("final          :", final_answer(result["messages"]))
    else:
        print("No Ollama reachable -- skipping the live run. The trace-reading above already works.")
except Exception as e:
    print("Live run skipped:", type(e).__name__)

---
### Key takeaway
The agent's loop is now visible: read tool_calls to see what it did, recursion_limit is your one-line guardrail. The message trace is your #1 debugging surface.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>